In [ ]:
import torch
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        """
        U-Net 모델 초기화

        Args:
            in_channels (int): 입력 이미지의 채널 수 (기본값: 1 - 그레이스케일)
            out_channels (int): 출력 세그멘테이션 맵의 채널 수 (기본값: 1 - 이진 분류)
        """
        super(UNet, self).__init__()

        def CBR(in_channels, out_channels):
            """
            Conv-BatchNorm-ReLU 블록 생성 함수

            U-Net의 기본 구성 단위로, 다음을 수행:
                1. 3x3 Convolution: 특징 추출
                2. Batch Normalization: 학습 안정화 및 수렴 속도 향상
                3. ReLU 활성화: 비선형성 추가

            Args:
                in_channels (int): 입력 채널 수
                out_channels (int): 출력 채널 수

            Returns:
                nn.Sequential: Conv2d → BatchNorm2d → ReLU로 구성된 레이어
            """
            layers = [
                # 3x3 합성곱 (padding=1로 크기 유지)
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
                # 배치 정규화: 각 미니배치의 평균과 분산을 정규화
                nn.BatchNorm2d(out_channels),
                # ReLU 활성화 함수: max(0, x)
                # inplace=True: 메모리 효율성 향상 (입력을 직접 수정)
                nn.ReLU(inplace=True)
            ]
            return nn.Sequential(*layers)

        # Contracting Path (인코더)
        # 이미지의 특징을 추출하며 공간 해상도는 줄이고 채널 수는 증가
        # 각 레벨에서 "무엇(What)"이 있는지 학습

        # 인코더 레벨 1: 1 → 64 채널
        # 입력: (batch, 1, H, W) → 출력: (batch, 64, H, W)
        self.enc1_1 = CBR(in_channels, 64)
        self.enc1_2 = CBR(64, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 해상도 1/2로 축소

        # 인코더 레벨 2: 64 → 128 채널
        # 입력: (batch, 64, H/2, W/2) → 출력: (batch, 128, H/2, W/2)
        self.enc2_1 = CBR(64, 128)
        self.enc2_2 = CBR(128, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 해상도 1/4로 축소

        # 인코더 레벨 3: 128 → 256 채널
        # 입력: (batch, 128, H/4, W/4) → 출력: (batch, 256, H/4, W/4)
        self.enc3_1 = CBR(128, 256)
        self.enc3_2 = CBR(256, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)  # 해상도 1/8로 축소

        # 인코더 레벨 4: 256 → 512 채널
        # 입력: (batch, 256, H/8, W/8) → 출력: (batch, 512, H/8, W/8)
        self.enc4_1 = CBR(256, 512)
        self.enc4_2 = CBR(512, 512)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)  # 해상도 1/16로 축소

        # Bottleneck (중심부)
        # 가장 깊은 레벨에서 고수준의 의미적 특징 추출
        # 입력: (batch, 512, H/16, W/16) → 출력: (batch, 1024, H/16, W/16)
        self.center1 = CBR(512, 1024)
        self.center2 = CBR(1024, 1024)  # 원 논문에서는 CBR을 2번 반복

        # Expansive Path (디코더)
        # 특징을 원본 이미지 크기로 복원하며 채널 수는 감소
        # 각 레벨에서 "어디(Where)"에 있는지 복원

        # 디코더 레벨 4: 1024 → 512 채널
        # Up-convolution: 해상도를 2배로 증가 (1/16 → 1/8)
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        # Skip connection 후: (512 from up4) + (512 from enc4) = 1024 채널
        self.dec4_1 = CBR(1024, 512)
        self.dec4_2 = CBR(512, 512)

        # 디코더 레벨 3: 512 → 256 채널
        # Up-convolution: 해상도를 2배로 증가 (1/8 → 1/4)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        # Skip connection 후: (256 from up3) + (256 from enc3) = 512 채널
        self.dec3_1 = CBR(512, 256)
        self.dec3_2 = CBR(256, 256)

        # 디코더 레벨 2: 256 → 128 채널
        # Up-convolution: 해상도를 2배로 증가 (1/4 → 1/2)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        # Skip connection 후: (128 from up2) + (128 from enc2) = 256 채널
        self.dec2_1 = CBR(256, 128)
        self.dec2_2 = CBR(128, 128)

        # 디코더 레벨 1: 128 → 64 채널
        # Up-convolution: 해상도를 2배로 증가 (1/2 → 1/1, 원본 크기)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        # Skip connection 후: (64 from up1) + (64 from enc1) = 128 채널
        self.dec1_1 = CBR(128, 64)
        self.dec1_2 = CBR(64, 64)

        # 최종 출력 레이어
        # 1x1 Convolution: 64채널을 out_channels로 변환
        # 각 픽셀에 대해 클래스별 점수(logits) 생성
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        """
        순전파 함수 - U-Net의 전체 연산 흐름

        Args:
            x (torch.Tensor): 입력 이미지 (batch_size, in_channels, H, W)

        Returns:
            torch.Tensor: 세그멘테이션 결과 (batch_size, out_channels, H, W)
                         각 픽셀의 클래스 점수(logits)
        """
        # Encoder (Contracting Path)
        # 레벨 1: 특징 추출
        enc1 = self.enc1_1(x)        # (B, 1, H, W) → (B, 64, H, W)
        enc1 = self.enc1_2(enc1)     # (B, 64, H, W) → (B, 64, H, W)
        pool1 = self.pool1(enc1)     # (B, 64, H, W) → (B, 64, H/2, W/2)

        # 레벨 2
        enc2 = self.enc2_1(pool1)    # (B, 64, H/2, W/2) → (B, 128, H/2, W/2)
        enc2 = self.enc2_2(enc2)     # (B, 128, H/2, W/2) → (B, 128, H/2, W/2)
        pool2 = self.pool2(enc2)     # (B, 128, H/2, W/2) → (B, 128, H/4, W/4)

        # 레벨 3
        enc3 = self.enc3_1(pool2)    # (B, 128, H/4, W/4) → (B, 256, H/4, W/4)
        enc3 = self.enc3_2(enc3)     # (B, 256, H/4, W/4) → (B, 256, H/4, W/4)
        pool3 = self.pool3(enc3)     # (B, 256, H/4, W/4) → (B, 256, H/8, W/8)

        # 레벨 4
        enc4 = self.enc4_1(pool3)    # (B, 256, H/8, W/8) → (B, 512, H/8, W/8)
        enc4 = self.enc4_2(enc4)     # (B, 512, H/8, W/8) → (B, 512, H/8, W/8)
        pool4 = self.pool4(enc4)     # (B, 512, H/8, W/8) → (B, 512, H/16, W/16)

        # Bottleneck
        center = self.center1(pool4)  # (B, 512, H/16, W/16) → (B, 1024, H/16, W/16)
        center = self.center2(center) # (B, 1024, H/16, W/16) → (B, 1024, H/16, W/16)

        # Decoder (Expansive Path) + Skip Connections
        # 레벨 4: 업샘플링 + 스킵 연결
        up4 = self.up4(center)                    # (B, 1024, H/16, W/16) → (B, 512, H/8, W/8)
        merge4 = torch.cat([up4, enc4], dim=1)    # (B, 512+512, H/8, W/8) = (B, 1024, H/8, W/8)
        dec4 = self.dec4_1(merge4)                # (B, 1024, H/8, W/8) → (B, 512, H/8, W/8)
        dec4 = self.dec4_2(dec4)                  # (B, 512, H/8, W/8) → (B, 512, H/8, W/8)

        # 레벨 3
        up3 = self.up3(dec4)                      # (B, 512, H/8, W/8) → (B, 256, H/4, W/4)
        merge3 = torch.cat([up3, enc3], dim=1)    # (B, 256+256, H/4, W/4) = (B, 512, H/4, W/4)
        dec3 = self.dec3_1(merge3)                # (B, 512, H/4, W/4) → (B, 256, H/4, W/4)
        dec3 = self.dec3_2(dec3)                  # (B, 256, H/4, W/4) → (B, 256, H/4, W/4)

        # 레벨 2
        up2 = self.up2(dec3)                      # (B, 256, H/4, W/4) → (B, 128, H/2, W/2)
        merge2 = torch.cat([up2, enc2], dim=1)    # (B, 128+128, H/2, W/2) = (B, 256, H/2, W/2)
        dec2 = self.dec2_1(merge2)                # (B, 256, H/2, W/2) → (B, 128, H/2, W/2)
        dec2 = self.dec2_2(dec2)                  # (B, 128, H/2, W/2) → (B, 128, H/2, W/2)

        # 레벨 1
        up1 = self.up1(dec2)                      # (B, 128, H/2, W/2) → (B, 64, H, W)
        merge1 = torch.cat([up1, enc1], dim=1)    # (B, 64+64, H, W) = (B, 128, H, W)
        dec1 = self.dec1_1(merge1)                # (B, 128, H, W) → (B, 64, H, W)
        dec1 = self.dec1_2(dec1)                  # (B, 64, H, W) → (B, 64, H, W)

        # 최종 출력
        # 1x1 Convolution으로 세그멘테이션 맵 생성
        out = self.final(dec1)  # (B, 64, H, W) → (B, out_channels, H, W)

        return out

In [ ]:
# 모델 테스트 및 정보 출력
print('🔧 U-Net 모델 생성 중...')
model = UNet(in_channels=1, out_channels=1)
print('✅ 모델 생성 완료!\n')

# 더미 입력으로 모델 테스트
dummy_input = torch.randn(2, 1, 512, 512)
output = model(dummy_input)

# 모델 정보 출력
print('📊 모델 정보:')
print(f'  - 입력 크기: {dummy_input.shape}  (batch, channels, height, width)')
print(f'  - 출력 크기: {output.shape}')
print(f'  - 총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}개')
print(f'  - 학습 가능한 파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}개')